# Unit 7 - 02 - Evaluate with SSIM

Similar to the pix2pixHD / GAN unit: sample a high-res estimate for each
test patch, compute **SSIM** against the ground-truth HR
plus the best / worst cases.


In [ ]:
import os, csv
import numpy as np
import tifffile
import torch
from torch.utils.data import Dataset, DataLoader
from skimage.transform import resize
from skimage.metrics import structural_similarity as ssim
from diffusers import UNet2DModel, DDIMScheduler
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
MARKER   = "CELL2"
MANIFEST = "./prepared/pairs_x4_%s.csv" % MARKER
CKPT     = "./checkpoints/sr3_caco2_x4.pt"
N_EVAL   = 100          # number of test patches to evaluate (set None for all)
DDIM_STEPS = 50

ckpt = torch.load(CKPT, map_location=device)
IMG_SIZE, TIMESTEPS = ckpt["img_size"], ckpt["timesteps"]

model = UNet2DModel(
    sample_size=IMG_SIZE, in_channels=2, out_channels=1, layers_per_block=2,
    block_out_channels=(64, 128, 256, 256),
    down_block_types=("DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
).to(device)
model.load_state_dict(ckpt["state_dict"]); model.eval()
print("loaded", CKPT, "| IMG_SIZE", IMG_SIZE)


In [ ]:
class SRCaco2Pairs(Dataset):
    def __init__(self, manifest, split, img_size):
        with open(manifest) as f:
            self.rows = [r for r in csv.DictReader(f) if r["split"] == split]
        self.img_size = img_size
    def __len__(self): return len(self.rows)
    @staticmethod
    def _load(path):
        img = tifffile.imread(path)
        if img.ndim == 3:
            img = img[..., 0] if img.shape[-1] <= 4 else img[0]
        return np.asarray(img, dtype=np.float32)
    def _prep(self, img):
        if img.shape[0] != self.img_size:
            img = resize(img, (self.img_size, self.img_size), order=3,
                         preserve_range=True, anti_aliasing=True)
        return img / 127.5 - 1.0
    def __getitem__(self, i):
        r = self.rows[i]
        lr = torch.from_numpy(self._prep(self._load(r["lr_path"])))[None].float()
        hr = torch.from_numpy(self._prep(self._load(r["hr_path"])))[None].float()
        return lr, hr

test_ds = SRCaco2Pairs(MANIFEST, "test", IMG_SIZE)
if N_EVAL:
    test_ds.rows = test_ds.rows[:N_EVAL]
test_dl = DataLoader(test_ds, batch_size=16, shuffle=False)
print("evaluating on", len(test_ds), "test patches")


In [ ]:
@torch.no_grad()
def sample(lr_img, num_steps=DDIM_STEPS):
    ddim = DDIMScheduler(num_train_timesteps=TIMESTEPS)
    ddim.set_timesteps(num_steps)
    x = torch.randn_like(lr_img)
    for t in ddim.timesteps:
        noise_pred = model(torch.cat([x, lr_img], dim=1), t).sample
        x = ddim.step(noise_pred, t, x).prev_sample
    return x.clamp(-1, 1)

def to01(x): return np.clip((x + 1) / 2, 0, 1)

ssim_model, ssim_lr, examples = [], [], []
for lr_b, hr_b in tqdm(test_dl, desc="sampling"):
    lr_b = lr_b.to(device)
    gen = sample(lr_b).cpu().numpy()
    lr_np, hr_np = lr_b.cpu().numpy(), hr_b.numpy()
    for i in range(gen.shape[0]):
        g, h, l = to01(gen[i, 0]), to01(hr_np[i, 0]), to01(lr_np[i, 0])
        sm = ssim(g, h, data_range=1.0)
        sl = ssim(l, h, data_range=1.0)
        ssim_model.append(sm); ssim_lr.append(sl)
        examples.append((l, g, h, sm))

ssim_model, ssim_lr = np.array(ssim_model), np.array(ssim_lr)
print("SSIM (LR input vs HR)   : %.4f" % ssim_lr.mean())
print("SSIM (generated vs HR)  : %.4f +/- %.4f" % (ssim_model.mean(), ssim_model.std()))


In [ ]:
# SSIM distribution
plt.figure(figsize=(6, 3))
plt.hist(ssim_model, bins=30, color="tab:blue", alpha=0.8)
plt.axvline(ssim_model.mean(), color="k", ls="--", label="mean %.3f" % ssim_model.mean())
plt.xlabel("SSIM (generated vs HR)"); plt.ylabel("count"); plt.legend()
plt.title("SR-CACO-2 X4 - per-patch SSIM"); plt.show()


In [ ]:
# Best / worst examples by SSIM
order = np.argsort([e[3] for e in examples])
picks = list(order[-3:][::-1]) + list(order[:3])    # 3 best, 3 worst
titles = ["best"] * 3 + ["worst"] * 3

fig, ax = plt.subplots(len(picks), 3, figsize=(8, 2.6 * len(picks)))
for row, (idx, tag) in enumerate(zip(picks, titles)):
    l, g, h, s = examples[idx]
    ax[row, 0].imshow(l, cmap="gray"); ax[row, 0].set_title("%s | LR input" % tag)
    ax[row, 1].imshow(g, cmap="gray"); ax[row, 1].set_title("generated (SSIM %.3f)" % s)
    ax[row, 2].imshow(h, cmap="gray"); ax[row, 2].set_title("HR target")
    for a in ax[row]: a.axis("off")
plt.tight_layout(); plt.show()


## Notes

* SSIM is reported the same way as the pix2pixHD / GAN unit
* Diffusion sampling is stochastic, re-running gives slightly different SSIM. For a fixed
  number, set a manual seed before `sample`, or average over a few seeds.
* This is intentionally minimal. Try to improve the code,
  raise `IMG_SIZE` toward the native 512, try to increase the number of EPOCHS
